In [6]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import time

# Webdriver inicializálása
driver = webdriver.Chrome()  # vagy Firefox, Edge stb.

try:
    # Oldal megnyitása
    url = "https://www.whoscored.com/matches/1914026/live/spain-laliga-2025-2026-real-betis-barcelona"
    driver.get(url)
    
    wait = WebDriverWait(driver, 20)
    
    # 1. Lépés: Cookie elfogadás gomb keresése és kattintás
    try:
        print("Cookie gomb keresése...")
        
        # Várjunk egy kicsit a cookie ablak megjelenésére
        time.sleep(2)
        
        # Próbáljuk megkeresni a gombot többféle módon
        cookie_button = None
        
        # 1. módszer: CSS selector a class és szöveg alapján
        try:
            cookie_button = wait.until(EC.element_to_be_clickable(
                (By.CSS_SELECTOR, 'button.Button__StyledButton-buoy__sc-a1qza5-0.elJono')
            ))
        except:
            # 2. módszer: Szöveg alapján (böngésző nyelvétől függően)
            try:
                cookie_button = wait.until(EC.element_to_be_clickable(
                    (By.XPATH, "//button[contains(text(), 'Az összes elfogadása')]")
                ))
            except:
                # 3. módszer: Button text részleges egyezés
                try:
                    cookie_button = wait.until(EC.element_to_be_clickable(
                        (By.XPATH, "//button[contains(., 'elfogadása')]")
                    ))
                except:
                    # 4. módszer: Background color alapján (ha nem változik)
                    try:
                        cookie_button = wait.until(EC.element_to_be_clickable(
                            (By.CSS_SELECTOR, 'button[style*="background-color: rgb(87, 220, 28)"]')
                        ))
                    except:
                        print("Nem található cookie gomb, lehet, hogy nem jelenik meg vagy már eltűnt")
        
        if cookie_button:
            print("Cookie gomb megtalálva, kattintás...")
            # Görgetés az elemhez (ha szükséges)
            driver.execute_script("arguments[0].scrollIntoView(true);", cookie_button)
            
            # Kattintás
            cookie_button.click()
            print("Cookie elfogadva!")
            
            # Várjunk egy kicsit, hogy bezáruljon a cookie ablak
            time.sleep(1)
        
    except Exception as cookie_error:
        print(f"Hiba a cookie gomb kezelése közben: {cookie_error}")
        print("Folytatás a Chalkboard kereséssel...")
    
    # 2. Lépés: Chalkboard link keresése és kattintás
    try:
        print("Chalkboard link keresése...")
        
        # Várakozás a Chalkboard link megjelenésére
        chalkboard_link = None
        
        try:
            # 1. módszer: CSS selector az href attribútum alapján
            chalkboard_link = wait.until(EC.element_to_be_clickable(
                (By.CSS_SELECTOR, 'a[href="#chalkboard"]')
            ))
        except TimeoutException:
            try:
                # 2. módszer: Link szövege alapján
                chalkboard_link = wait.until(EC.element_to_be_clickable(
                    (By.LINK_TEXT, "Chalkboard")
                ))
            except TimeoutException:
                try:
                    # 3. módszer: XPath a span szövege alapján
                    chalkboard_link = wait.until(EC.element_to_be_clickable(
                        (By.XPATH, '//a[@href="#chalkboard"]//span[contains(text(), "Chalkboard")]')
                    ))
                except TimeoutException:
                    # 4. módszer: Általánosabb keresés
                    chalkboard_link = wait.until(EC.element_to_be_clickable(
                        (By.XPATH, '//a[contains(@href, "chalkboard") or contains(.//text(), "Chalkboard")]')
                    ))
        
        if chalkboard_link:
            print("Chalkboard link megtalálva, kattintás...")
            
            # Görgetés az elemhez
            driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", chalkboard_link)
            
            # Várakozás, hogy az elem tényleg kattintható legyen
            time.sleep(0.5)
            
            # Kattintás
            chalkboard_link.click()
            
            # Alternatív kattintás JavaScript-tel, ha a normál nem működik
            # driver.execute_script("arguments[0].click();", chalkboard_link)
            
            print("Sikeresen rákattintottam a Chalkboard linkre!")
            
            # Várjunk egy kicsit, hogy betöltődjön a tartalom
            time.sleep(2)
            
            # Ellenőrizzük, hogy a kattintás ténylegesen történt-e
            current_url = driver.current_url
            if "#chalkboard" in current_url:
                print("Sikeresen átváltott a Chalkboard fülre!")
            else:
                print("Megjegyzés: Az URL nem tartalmazza a '#chalkboard'-ot, de a kattintás megtörtént")
    
    except Exception as chalkboard_error:
        print(f"Hiba a Chalkboard link kezelése közben: {chalkboard_error}")

except Exception as e:
    print(f"Általános hiba történt: {str(e)}")

finally:
    # Opcionális: várakozás, hogy láthasd az eredményt
    print("\n" + "="*50)
    sleep = 2
    print(f"A böngésző nyitva marad {sleep} másodpercig...")
    print("Bezáráshoz nyomj Entert a konzolon, vagy zárd be kézzel.")
    print("="*50 + "\n")
    
    # Várakozás Enter-re a konzolon (opcionális)
    # input("Nyomj Entert a böngésző bezárásához...")
    
    # Automatikus bezárás 30 másodperc múlva
    time.sleep(sleep)
    driver.quit()

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import json
import time

def extract_event_data(driver):
    """
    JavaScript injection az események adatainak kinyerésére
    """
    # 1. Próbáld meg kinyerni a globális változókat
    scripts = [
        # A leggyakoribb helyek, ahol tárolhatják az eseményeket
        "return window.eventData || window.chalkboardData || window.matchEvents;",
        "return typeof eventsData !== 'undefined' ? eventsData : null;",
        "return document.chalkboardEvents || null;",
        # Canvas elem adatainak kinyerése (ha van hidden data attribútum)
        "const canvas = document.getElementById('undefinedStatsCanvas'); return canvas.dataset;",
    ]
    
    for script in scripts:
        try:
            result = driver.execute_script(script)
            if result:
                print(f"Adatok megtalálva: {result}")
                return result
        except:
            continue
    
    # 2. Próbáld meg a hálózati kérések adatait
    try:
        network_data = driver.execute_script("""
            // Ha van localStorage vagy sessionStorage
            const data = {};
            for(let i = 0; i < localStorage.length; i++) {
                const key = localStorage.key(i);
                if(key.includes('event') || key.includes('chalkboard')) {
                    try {
                        data[key] = JSON.parse(localStorage.getItem(key));
                    } catch(e) {
                        data[key] = localStorage.getItem(key);
                    }
                }
            }
            return data;
        """)
        if network_data:
            print(f"LocalStorage adatok: {json.dumps(network_data, indent=2)}")
    except:
        pass
    
    return None

# Események kinyerése
event_data = extract_event_data(driver)

Cookie gomb keresése...
Cookie gomb megtalálva, kattintás...
Cookie elfogadva!
Chalkboard link keresése...
Chalkboard link megtalálva, kattintás...
Sikeresen rákattintottam a Chalkboard linkre!
Megjegyzés: Az URL nem tartalmazza a '#chalkboard'-ot, de a kattintás megtörtént

A böngésző nyitva marad 2 másodpercig...
Bezáráshoz nyomj Entert a konzolon, vagy zárd be kézzel.

